# mp-spawn-workers — faded example 1: Write a worker to a temp file and import it for mp.spawn

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`. Running the beacon reports progress on the `Distributed: mp.spawn workers` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: mp.spawn workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mp-spawn-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mp-spawn-workers"
DD_SUBTOPIC = "Distributed: mp.spawn workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`mp.spawn` requires the worker function to be importable by child processes. Since Jupyter/Colab cells define functions in `__main__`, which is not importable, the workaround is to write the worker source to a `.py` file in `/tmp`, add `/tmp` to `sys.path`, and import the module with `importlib`. The reload guard (`importlib.reload` if already in `sys.modules`) handles repeated cell executions.

## Faded exercise 1

Implement `load_worker_module(src_str, module_name)` that:
1. Writes `src_str` to `/tmp/<module_name>.py`.
2. Adds `/tmp` to `sys.path` if not already present.
3. If `module_name` is already in `sys.modules`, reloads it; otherwise imports it fresh.
4. Returns the imported module.

Your task: **fill in the import-or-reload logic (step 3)**.

**Fill in:** The sys.modules check that either reloads an already-cached module with importlib.reload or imports it fresh with importlib.import_module, then assigns the result to mod.

In [ ]:
import sys
import importlib
import importlib.util
import os

def load_worker_module(src_str: str, module_name: str):
    path = f'/tmp/{module_name}.py'
    with open(path, 'w') as f:
        f.write(src_str)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    # Drop any stale compiled bytecode and the finder cache so that a
    # rewrite within the same second is actually re-read from source.
    try:
        os.remove(importlib.util.cache_from_source(path))
    except OSError:
        pass
    importlib.invalidate_caches()
    if module_name in sys.modules:
        mod = importlib.reload(sys.modules[module_name])
    else:
        mod = importlib.import_module(module_name)
    return mod

def _test():
    import sys, importlib
    src = 'def hello(rank, world_size): return rank * world_size\n'
    mod = load_worker_module(src, 'dd_faded_test_mod')
    assert hasattr(mod, 'hello')
    assert mod.hello(2, 5) == 10
    mod2 = load_worker_module(src, 'dd_faded_test_mod')
    assert mod2.hello(3, 4) == 12


def _test():
    import sys, importlib
    src = 'def add(rank, world_size): return rank + world_size\n'
    mod = load_worker_module(src, 'dd_faded_add_mod')
    assert hasattr(mod, 'add'), "module should have add fn"
    assert mod.add(1, 4) == 5
    # Reload path: update the function and reload
    src2 = 'def add(rank, world_size): return rank * world_size\n'
    mod2 = load_worker_module(src2, 'dd_faded_add_mod')
    assert mod2.add(3, 4) == 12, "after reload should use new src"
    # /tmp should be in sys.path
    assert '/tmp' in sys.path


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import sys
import importlib
import importlib.util
import os

def load_worker_module(src_str: str, module_name: str):
    path = f'/tmp/{module_name}.py'
    with open(path, 'w') as f:
        f.write(src_str)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    # Drop any stale compiled bytecode and the finder cache so that a
    # rewrite within the same second is actually re-read from source.
    try:
        os.remove(importlib.util.cache_from_source(path))
    except OSError:
        pass
    importlib.invalidate_caches()
    if module_name in sys.modules:
        mod = importlib.reload(sys.modules[module_name])
    else:
        mod = importlib.import_module(module_name)
    return mod

def _test():
    import sys, importlib
    src = 'def hello(rank, world_size): return rank * world_size\n'
    mod = load_worker_module(src, 'dd_faded_test_mod')
    assert hasattr(mod, 'hello')
    assert mod.hello(2, 5) == 10
    mod2 = load_worker_module(src, 'dd_faded_test_mod')
    assert mod2.hello(3, 4) == 12
```
</details>